In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

os.listdir('/content')

['.config', '2015-summary.csv', 'sample_data']

In [ ]:
pip install pyspark

In [ ]:
import pyspark

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark=SparkSession.builder.getOrCreate()

In [ ]:
flightData2015 = spark.read.csv(
    "/content/2015-summary.csv",
    header=True,  # Tells Spark that the first line of the file holds column names, not data.
    inferSchema=True # Tells Spark to scan the data and guess each column's type , Without it: everything is read as string, even count, so sum("count") or sorting would behave badly
)

In [ ]:
flightData2015.take(5)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Croatia', count=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Ireland', count=344),
 Row(DEST_COUNTRY_NAME='Egypt', ORIGIN_COUNTRY_NAME='United States', count=15),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='India', count=62)]

In [ ]:
flightData2015.sort("count").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [count#19 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(count#19 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=33]
      +- FileScan csv [DEST_COUNTRY_NAME#17,ORIGIN_COUNTRY_NAME#18,count#19] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/2015-summary.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:int>




# **RDD**

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext # old version

In [ ]:
 # Alternative way to import sparkContext , This line creates the SparkContext, the entry point to Spark's low-level (RDD) API

from pyspark import SparkContext
sc = SparkContext() # This line creates the SparkContext, the entry point to Spark's low-level (RDD) API

# 1- create an RDD from a collection

In [ ]:
data = list(range(10))

In [ ]:
data

In [ ]:
rdd1 = sc.parallelize(data) # split it to partition

In [ ]:
rdd1    # it doesn't contain any data yet

ParallelCollectionRDD[1] at readRDDFromFile at PythonRDD.scala:298

In [ ]:
rdd1.getNumPartitions()

2

## Map

In [ ]:
rdd2 = rdd1.map(lambda x : x**2)

In [ ]:
rdd3 = rdd2.map(lambda x : x+5)

In [ ]:
rdd4 = rdd2.map(lambda x : x-5)

In [ ]:
rdd5 = rdd2.map(lambda x : x/2)

In [ ]:
rdd2   # it doesn't contain any data yet

PythonRDD[2] at RDD at PythonRDD.scala:57

In [ ]:
rdd2.persist() # when excute , save the result

PythonRDD[2] at RDD at PythonRDD.scala:57

In [ ]:
rdd2.collect()

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

In [ ]:
rdd3.collect()

[5, 6, 9, 14, 21, 30, 41, 54, 69, 86]

In [ ]:
rdd4.collect()

[-5, -4, -1, 4, 11, 20, 31, 44, 59, 76]

# Note

In [ ]:
rdd6 = rdd2.map(lambda x : x/0)      # error

In [ ]:
rdd6.take(2)

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 3.0 failed 1 times, most recent failure: Lost task 0.0 in stage 3.0 (TID 6) (ad2dd14f57a4 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 2044, in main
    process()
    ~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 2036, in process
    serializer.dump_stream(out_iter, outfile)
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 273, in dump_stream
    vs = list(itertools.islice(iterator, batch))
  File "/usr/local/lib/python3.13/dist-packages/pyspark/core/rdd.py", line 2716, in takeUpToNum

## lambda

In [ ]:
# creating function

def sq(no):
    return no**2

In [ ]:
sq(3)

In [ ]:
#function written as an expresstion

sq_lambda  = lambda no:no**2

In [ ]:
sq_lambda(3)

In [ ]:
rdd7 = rdd2.map(sq)

# 2- create an RDD from a text file

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving example2.txt to example2.txt


In [ ]:
rdd = sc.textFile("/content/example2.txt")

In [ ]:
rdd

/content/example2.txt MapPartitionsRDD[7] at textFile at NativeMethodAccessorImpl.java:0

In [ ]:
rdd.getNumPartitions()

2

In [ ]:
rdd.collect()  # any action returns a list

['first ', 'second line', 'the third line', 'then a fourth line']

In [ ]:
rdd2 = rdd.map(lambda line : line.split()) #return list of lists

In [ ]:
rdd2

PythonRDD[8] at RDD at PythonRDD.scala:57

In [ ]:
rdd2.collect()

[['first'],
 ['second', 'line'],
 ['the', 'third', 'line'],
 ['then', 'a', 'fourth', 'line']]

## flatMap

In [ ]:
rdd3 = rdd.flatMap(lambda line : line.split())

In [ ]:
rdd3.collect()

['first',
 'second',
 'line',
 'the',
 'third',
 'line',
 'then',
 'a',
 'fourth',
 'line']